<a href="https://colab.research.google.com/github/Saivamshi-K/Mutanews-Beyond-Static-Detection-of-Fake-News/blob/main/Forecasting_cyber_crimes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-learn tensorflow matplotlib seaborn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Simulated SCI dataset
data_pre = {
    "SCI_Type": ["Malware", "DDoS", "APT", "Phishing", "Malware", "SQL Injection", "Espionage", "DDoS"] * 63,
    "Continent": ["Asia", "Europe", "North America", "Asia", "Europe", "Africa", "South America", "Asia"] * 63
}
data_post = {
    "SCI_Type": ["Phishing", "Malware", "APT", "DDoS", "SQL Injection", "Espionage", "Zero-Day Exploit", "Malware"] * 68,
    "Continent": ["Asia", "Europe", "Asia", "North America", "Africa", "Europe", "Asia", "South America"] * 68
}

df_pre = pd.DataFrame(data_pre)
df_post = pd.DataFrame(data_post)

# TF-IDF vectorization
vectorizer = TfidfVectorizer()
X_pre = vectorizer.fit_transform(df_pre["SCI_Type"])
X_post = vectorizer.transform(df_post["SCI_Type"])

# Label encoding
le = LabelEncoder()
y_pre = le.fit_transform(df_pre["Continent"])
y_post = le.transform(df_post["Continent"])

# Split datasets
X_train_pre, X_test_pre, y_train_pre, y_test_pre = train_test_split(X_pre, y_pre, test_size=0.3, random_state=42)
X_train_post, X_test_post, y_train_post, y_test_post = train_test_split(X_post, y_post, test_size=0.3, random_state=42)

# Classifiers
models = {
    "SVM": SVC(kernel='linear'),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

results = {"Model": [], "Pre-Pandemic Accuracy": [], "Post-Pandemic Accuracy": []}

for name, model in models.items():
    model.fit(X_train_pre, y_train_pre)
    y_pred_pre = model.predict(X_test_pre)
    acc_pre = accuracy_score(y_test_pre, y_pred_pre)

    model.fit(X_train_post, y_train_post)
    y_pred_post = model.predict(X_test_post)
    acc_post = accuracy_score(y_test_post, y_pred_post)

    results["Model"].append(name)
    results["Pre-Pandemic Accuracy"].append(round(acc_pre * 100, 2))
    results["Post-Pandemic Accuracy"].append(round(acc_post * 100, 2))

df_results = pd.DataFrame(results)
print(df_results)

# Plot
plt.figure(figsize=(10, 6))
bar_width = 0.35
index = np.arange(len(df_results))
plt.bar(index, df_results["Pre-Pandemic Accuracy"], bar_width, label='Pre-Pandemic', color='skyblue')
plt.bar(index + bar_width, df_results["Post-Pandemic Accuracy"], bar_width, label='Post-Pandemic', color='salmon')
plt.xlabel('Classifier')
plt.ylabel('Accuracy (%)')
plt.title('Figure 4: Accuracy Comparison of Classifiers on SCI Datasets')
plt.xticks(index + bar_width / 2, df_results["Model"])
plt.ylim(80, 100)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

# Simulated Annual SCI Counts (2003–2025)
years = np.arange(2003, 2026)
sci_counts = np.array([
    15, 18, 22, 25, 28, 30, 34, 38, 42, 47, 53, 59, 65, 71, 80, 87,
    93, 101, 108, 114, 127, 135, 142
])

df = pd.DataFrame({'Year': years, 'SCI_Count': sci_counts})

# Normalize
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df[['SCI_Count']])

# Create sequences
def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

sequence_length = 3
X, y = create_sequences(data_scaled, sequence_length)

# Build LSTM model
model = Sequential([
    LSTM(50, activation='relu', input_shape=(sequence_length, 1)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.fit(X, y, epochs=200, verbose=0)

# Forecast next 3 steps
forecast_steps = 3
input_seq = data_scaled[-sequence_length:].reshape(1, sequence_length, 1)
predictions = []

for _ in range(forecast_steps):
    pred = model.predict(input_seq, verbose=0)[0]
    predictions.append(pred)
    input_seq = np.append(input_seq[:, 1:, :], [[pred]], axis=1)

predicted_counts = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()
future_years = np.arange(2026, 2026 + forecast_steps)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(years, sci_counts, label="Historical SCI Count", marker='o')
plt.plot(future_years, predicted_counts, label="LSTM Forecast", marker='x', linestyle='--', color='red')
plt.title("Figure 3: Annual SCI Counts with LSTM Forecast (2003–2028)")
plt.xlabel("Year")
plt.ylabel("SCI Count")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)

pre_sci_data = {
    "Asia": ["DDoS on telecom", "SQL injection", "Phishing emails", "Malware in Indian banks", "Botnet in China"],
    "Europe": ["Banking trojan in France", "Spyware in German servers", "Insider breach in UK", "Data leak in EU"],
    "North America": ["Credential theft in US", "Keylogger in Canadian government", "Social engineering in universities"],
    "Africa": ["Email spoofing in Nigeria", "Brute force attack in Kenya", "Phishing in mobile networks"],
    "South America": ["Trojan in Brazil banks", "SQL attack on Argentina tax portal", "Ransomware in Colombia"],
    "Oceania": ["DNS misconfig in AU", "Trojan dropper in NZ", "DDoS in healthcare infra"]
}

def generate_improved_samples(sci_dict, count_per_class=100):
    samples, labels = [], []
    for region, texts in sci_dict.items():
        for _ in range(count_per_class):
            text = np.random.choice(texts)
            if np.random.rand() < 0.01:
                wrong_region = np.random.choice([r for r in sci_dict if r != region])
                labels.append(wrong_region)
            else:
                labels.append(region)
            samples.append(text)
    return pd.DataFrame({"SCI_Type": samples, "Continent": labels})

df_pre_improved = generate_improved_samples(pre_sci_data, count_per_class=100)

le = LabelEncoder()
y = le.fit_transform(df_pre_improved["Continent"])
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df_pre_improved["SCI_Type"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svm = SVC(kernel='linear', probability=True)
rf = RandomForestClassifier()
nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)

ensemble = VotingClassifier(estimators=[('svm', svm), ('rf', rf), ('nb', nb), ('lr', lr)], voting='hard')
ensemble.fit(X_train, y_train)
preds = ensemble.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"Improved Pre-Pandemic Ensemble Accuracy: {acc * 100:.2f}%")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)

post_sci_data = {
    "Asia": [
        "Zero-Day attack in supply chain vendor", "IoT malware in telecoms",
        "Advanced persistent threat on payment systems", "Ransomware in Indian education"
    ],
    "Europe": [
        "Ransomware in EU hospitals", "AI-based deepfake phishing attack",
        "Cloud misconfig in government", "Spyware in French tax records"
    ],
    "North America": [
        "Infrastructure breach in US pipelines", "Software backdoor in Canadian company",
        "Credential leak in cloud services", "Phishing on remote learning platforms"
    ],
    "Africa": ["Mobile banking trojan in Nigeria", "Crypto scam in Ghana", "Phishing on eWallet apps"],
    "South America": ["COVID-themed phishing in Brazil", "Fake vaccine sites in Chile", "Supply chain ransomware"],
    "Oceania": ["DDoS on Australian hospitals", "Malware in NZ finance", "Remote work attack on AU gov"]
}

def generate_improved_post_samples(sci_dict, count_per_class=100):
    samples, labels = [], []
    for region, texts in sci_dict.items():
        for _ in range(count_per_class):
            text = np.random.choice(texts)
            if np.random.rand() < 0.01:
                wrong_region = np.random.choice([r for r in sci_dict if r != region])
                labels.append(wrong_region)
            else:
                labels.append(region)
            samples.append(text)
    return pd.DataFrame({"SCI_Type": samples, "Continent": labels})

df_post_improved = generate_improved_post_samples(post_sci_data, count_per_class=100)

le = LabelEncoder()
y = le.fit_transform(df_post_improved["Continent"])
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df_post_improved["SCI_Type"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svm = SVC(kernel='linear', probability=True)
rf = RandomForestClassifier()
nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)

ensemble = VotingClassifier(estimators=[('svm', svm), ('rf', rf), ('nb', nb), ('lr', lr)], voting='hard')
ensemble.fit(X_train, y_train)
preds = ensemble.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"Improved Post-Pandemic Ensemble Accuracy: {acc * 100:.2f}%")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)

sci_data = {
    "Asia": [
        "APT attack on telecom", "Malware in banking", "Phishing via email",
        "DDoS on cloud infra", "Credential theft from eCommerce", "Zero-Day in routers"
    ],
    "Europe": [
        "SQL injection on public portal", "Espionage in defense systems",
        "Malware via USB", "Phishing at EU institutions", "Trojan in business networks",
        "Spyware in email attachments"
    ],
    "North America": [
        "Ransomware in US hospitals", "Data breach in universities",
        "Supply chain attack on software vendor", "Credential stuffing",
        "APT on government servers", "Malware in financial system"
    ],
    "Africa": [
        "DNS hijack in Nigeria", "Phishing through mobile apps",
        "Man-in-the-middle over Wi-Fi", "Brute force login", "Trojan from pirated software"
    ],
    "South America": [
        "Banking trojan in Brazil", "Fake antivirus scam", "Ransomware in oil industry",
        "Data theft in government cloud", "SQL injection on tax site"
    ],
    "Oceania": [
        "DDoS on AU stock exchange", "Spyware in NZ finance sector",
        "Malware in healthcare", "Social engineering via SMS", "Credential phishing email"
    ]
}

samples, labels = [], []
for label, types in sci_data.items():
    for _ in range(140):
        sci_text = np.random.choice(types)
        if np.random.rand() < 0.02:
            noisy_label = np.random.choice([l for l in sci_data if l != label])
            labels.append(noisy_label)
        else:
            labels.append(label)
        samples.append(sci_text)

df = pd.DataFrame({"SCI_Type": samples, "Continent": labels})

le = LabelEncoder()
y = le.fit_transform(df["Continent"])
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df["SCI_Type"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svm = SVC(kernel='linear', probability=True)
rf = RandomForestClassifier()
nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)

ensemble = VotingClassifier(estimators=[('svm', svm), ('rf', rf), ('nb', nb), ('lr', lr)], voting='hard')
ensemble.fit(X_train, y_train)
ensemble_preds = ensemble.predict(X_test)
ensemble_acc = accuracy_score(y_test, ensemble_preds)
print(f"Ensemble Accuracy: {ensemble_acc * 100:.2f}%")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

pre_sci_data = {
    "Asia": ["DDoS attack on telecom", "SQL injection attempt", "Phishing email detected", "Malware in Indian banks", "Botnet in China"],
    "Europe": ["Banking trojan in France", "Spyware in German servers", "Insider breach in UK", "Data leak in EU"],
    "North America": ["Credential theft in US", "Keylogger in government systems", "Social engineering attack"],
    "Africa": ["Email spoofing in Nigeria", "Brute force attack in Kenya", "Phishing in mobile networks"],
    "South America": ["Trojan in Brazil banks", "SQL attack on tax portal", "Ransomware in Colombia"],
    "Oceania": ["DNS misconfig in AU", "Trojan dropper in NZ", "DDoS in healthcare infra"]
}

def generate_varied_samples(data_dict, n_per_class=150):
    texts, labels = [], []
    for label, samples in data_dict.items():
        for _ in range(n_per_class):
            base = np.random.choice(samples)
            noise = np.random.choice(["", " reported", " case", " attempt", " warning", " issue", " alert"])
            typo = np.random.choice(["", "!", ".", " "])
            texts.append(base + noise + typo)
            labels.append(label)
    return pd.DataFrame({"SCI_Type": texts, "Continent": labels})

df_pre = generate_varied_samples(pre_sci_data)

texts = df_pre["SCI_Type"].values
labels = df_pre["Continent"].values

le = LabelEncoder()
y = le.fit_transform(labels)
y = to_categorical(y)

tokenizer = Tokenizer(num_words=2000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, maxlen=15)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

vocab_size = len(tokenizer.word_index) + 1

model = Sequential([
    Embedding(vocab_size, 64, input_length=15),
    Conv1D(32, 3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.5),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(y.shape[1], activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=6, batch_size=32, validation_split=0.2, verbose=1)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Pre-Pandemic CNN Accuracy: {acc * 100:.2f}%")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.utils import to_categorical

post_sci_data = {
    "Asia": [
        "Zero-Day attack in supply chain vendor", "IoT malware in telecoms",
        "Advanced persistent threat on payment systems", "Ransomware in Indian education"
    ],
    "Europe": [
        "Ransomware in EU hospitals", "AI-based deepfake phishing attack",
        "Cloud misconfig in government", "Spyware in French tax records"
    ],
    "North America": [
        "Infrastructure breach in US pipelines", "Software backdoor in Canadian company",
        "Credential leak in cloud services", "Phishing on remote learning platforms"
    ],
    "Africa": ["Mobile banking trojan in Nigeria", "Crypto scam in Ghana", "Phishing on eWallet apps"],
    "South America": ["COVID-themed phishing in Brazil", "Fake vaccine sites in Chile", "Supply chain ransomware"],
    "Oceania": ["DDoS on Australian hospitals", "Malware in NZ finance", "Remote work attack on AU gov"]
}

def generate_samples(data_dict, count_per_class=100):
    samples, labels = [], []
    for region, texts in data_dict.items():
        for _ in range(count_per_class):
            text = np.random.choice(texts)
            if np.random.rand() < 0.02:
                wrong_region = np.random.choice([r for r in data_dict if r != region])
                labels.append(wrong_region)
            else:
                labels.append(region)
            samples.append(text)
    return pd.DataFrame({"Text": samples, "Label": labels})

df = generate_samples(post_sci_data, 100)

le = LabelEncoder()
y = le.fit_transform(df["Label"])
y = to_categorical(y)

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["Text"])
X = tokenizer.texts_to_sequences(df["Text"])
maxlen = 15
X = pad_sequences(X, maxlen=maxlen)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vocab_size = len(tokenizer.word_index) + 1

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=maxlen),
    Conv1D(128, 3, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(y.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=12, batch_size=16, validation_split=0.2, verbose=0)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Post-Pandemic CNN Accuracy: {acc * 100:.2f}%")